In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from tqdm import tqdm

In [2]:
# 1. 장치 설정 (Mac M4 Pro 가속기 활성화)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 M4 Pro GPU (MPS) 활성화 완료!")
else:
    device = torch.device("cpu")

# 2. 데이터 전처리 (위성 이미지를 위한 특별 세팅)
# 위성에서 찍은 사진은 상하좌우가 뒤집혀도 동일한 지형이므로, 
# RandomHorizontalFlip과 RandomVerticalFlip을 모두 써주는 것이 95% 달성의 핵심입니다!
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. 데이터셋 로드 및 분할
# EuroSAT은 총 27,000장입니다. (Train 80%, Val 10%, Test 10%로 직접 나눕니다)
print("데이터셋을 다운로드하고 로드하는 중...")
full_dataset = torchvision.datasets.EuroSAT(root='./data', download=True, transform=transform)

train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# M4 Pro 48GB 메모리에 맞춘 시원한 배치 사이즈와 풀코어 로딩
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=8)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)

print(f"데이터 준비 완료! (학습: {len(train_dataset)}장, 검증: {len(val_dataset)}장, 테스트: {len(test_dataset)}장)")

# 4. 모델 정의 (EfficientNet-B0 사용)
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
num_ftrs = model.classifier[1].in_features
# EuroSAT은 총 10개의 토지 클래스(숲, 고속도로, 강 등)를 가집니다.
model.classifier[1] = nn.Linear(num_ftrs, 10) 
model = model.to(device)

# 5. 손실함수 및 최적화 도구
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002) # 큰 배치사이즈에 맞춘 학습률
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.1)

# 6. 학습 루프
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100 * correct / total
    
    # Validation 체크
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    print(f"Loss: {running_loss/len(train_loader):.4f} | Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    
    scheduler.step(val_loss)

# 7. 최종 테스트
print("\n--- Final Test ---")
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

print(f"최종 Test Accuracy: {100 * test_correct / test_total:.2f}%")

🚀 M4 Pro GPU (MPS) 활성화 완료!
데이터셋을 다운로드하고 로드하는 중...


100%|██████████████████████████████████████| 94.3M/94.3M [00:01<00:00, 56.4MB/s]


데이터 준비 완료! (학습: 21600장, 검증: 2700장, 테스트: 2700장)


Epoch 1/10: 100%|█████████████████████████████| 169/169 [02:49<00:00,  1.00s/it]


Loss: 0.2519 | Acc: 92.32% | Val Acc: 96.89%


...

Loss: 0.0208 | Acc: 99.29% | Val Acc: 98.63%


Epoch 9/10: 100%|█████████████████████████████| 169/169 [02:44<00:00,  1.03it/s]


Loss: 0.0190 | Acc: 99.40% | Val Acc: 98.78%


Epoch 10/10: 100%|████████████████████████████| 169/169 [02:46<00:00,  1.02it/s]


Loss: 0.0174 | Acc: 99.44% | Val Acc: 98.63%

--- Final Test ---
최종 Test Accuracy: 99.04%
